# **Airborne Object Tracking Introspection Module Evaluation**

## **1. Overview**

This notebook evaluates the introspective self-monitoring layer built on top of the HRNet-W32 Tracking-to-Detect (T2D) pipeline, for the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge), a benchmark designed for long-range airborne object detection and tracking in autonomous Sense & Avoid (SAA) systems. Autonomous SAA systems are responsible for maintaining safe separation between aircraft and unforeseen obstacles during flight. While flight paths may be planned in advance, real-world operation in unconstrained airspace introduces the possibility of encountering unexpected airborne objects. In such cases, the system must perform situational awareness, decision-making, and evasive manoeuvre planning in real time. 

The primary objective of this notebook is to evaluate **introspection strategies** layered on top of a fixed HRNet-W32 Tracking-to-Detect (T2D) backbone. Rather than comparing alternative detection backbones, each introspection strategy monitors the output of the existing tracking pipeline and determines, on a per-track and per-frame basis, whether tracking quality is degrading. The underlying detection and tracking pipeline remains unchanged throughout all experiments, ensuring that any differences in performance are attributable solely to the introspection module.

### **1.1. Example Airborne Encounter**

<div align="center">
  <img src="../docs/Images/Data_Viewer_Video.gif" width="700"/>
</div>

<p align="center">
  <em>Example Airborne Encounter from the Evaluation Dataset. The Target Aircraft Occupies Only a Few Pixels During the Early Stages of the Encounter.</em>
</p>


---

## **2. Introspection Modules Evaluated**

* **Default Condition:** Never raises a flag. Used to verify that the introspective wrapper produces results identical to the baseline T2D pipeline.
* **Simple Confidence Threshold:** Flags a track as unreliable when its detection confidence remains below a fixed threshold for a specified number of consecutive frames. Serves as the simplest interpretable baseline.

<div style="background-color:#fff8dc; border-left:5px solid #f0c419; padding:10px 15px; margin:10px 0;"> <strong>⚠️ PLACEHOLDER - Additional introspection modules will be added.</strong><br/> Update this section to include all implemented modules once development and evaluation are complete. </div>


---

## **3. Evaluation Metrics**

* **AFDR:** Average Frame Detection Rate.
* **CEDR:** Continuous Encounter Detection Rate.
* **FTPS:** False Tracks Per Sequence.

> ⚠️ ***Note:*** *Inference for all models requires a CUDA-capable GPU. While CPU-based loading of model weights is supported for inspection, full evaluation is computationally intensive.*

<div style="background-color:#fff8dc; border-left:5px solid #f0c419; padding:10px 15px; margin:10px 0;">
<strong>⚠️ PLACEHOLDER - Introspection-specific failure detection metrics are to be determined.</strong></br>
Populate this section once the evaluation protocol has been finalised.
</div>



---

## **4. Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

### **4.1. Example Detection Progression**
<div align="center" style="display: flex; justify-content: center; gap: 20px;">
  <div style="text-align: center;">
    <img src="../docs/Images/Data_Viewer_Picture_1.png" width="650"/>
    <p><em>Early Detection</em></p>
  </div>
  <div style="text-align: center;">
    <img src="../docs/Images/Data_Viewer_Picture_2.png" width="650"/>
    <p><em>Closing Distance</em></p>
  </div>
  <div style="text-align: center;">
    <img src="../docs/Images/Data_Viewer_Picture_3.png" width="650"/>
    <p><em>Near Range</em></p>
  </div>
</div>

*The sequence above demonstrates how airborne targets gradually increase in apparent size throughout an encounter. Reliable early detection is particularly challenging because aircraft often occupy only a few pixels during the most safety-critical phase of the encounter.*

<br/>

### **4.2. Hard Subset Selection Criteria**

- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

<br/>

### **4.3. Dataset Summary**

The dataset is partitioned into a stratified 70/15/15 train/validation/test split to preserve the distribution of key difficulty factors across all subsets.

<div align="center">

<table>
  <thead>
    <tr>
      <th>Metric</th>
      <th>Value</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Total Flights</strong></td>
      <td><strong>1618</strong></td>
      <td><strong>Total number of flight sequences in the dataset.</strong></td>
    </tr>
    <tr>
      <td>Small Object Flights</td>
      <td>662</td>
      <td>Flights containing at least one object with bounding box area ≤ 36.0px².</td>
    </tr>
    <tr>
      <td>Below Horizon Flights</td>
      <td>1053</td>
      <td>Flights containing at least one object at or below the horizon.</td>
    </tr>
    <tr>
      <td>Multi-Object Flights</td>
      <td>627</td>
      <td>Flights containing more than one airborne object in at least one frame.</td>
    </tr>
    <tr>
      <td>No Object Frames</td>
      <td>354</td>
      <td>Flights containing at least one frame with no airborne object present.</td>
    </tr>
    <tr>
      <td><em>Avg. Sequence Length</em></td>
      <td><em>17.77s</em></td>
      <td><em>Average duration of each flight sequence.</em></td>
    </tr>
  </tbody>
</table>

</div>


<div style="background-color:#fff8dc; border-left:5px solid #f0c419; padding:10px 15px; margin:10px 0;">
<strong>⚠️ PLACEHOLDER - Evaluation currently uses the unmodified AOT held-out test set. </strong><br/>
A controlled degradation benchmark, based on the AOT-C corruption protocol, remains under consideration and has not yet been implemented. Refer to the project notes for additional discussion.
</div>


---

<br/>

In [1]:
# Import Required Libraries.
import os
import sys

import cv2
import json
import torch
import random

from glob import glob
import matplotlib.pyplot as plt

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.data.original_dataset.download_data import main as downloadDataset
from src.data.corruption_dataset.build_corrupted_test_set import main as corruptDataset
from src.introspection_module.view_introspection_results import generateIntrospectionVideo, spliceVideosSideBySide
from notebooks.helpers.notebook_helpers import (
    get_next_run_dir, load_combined_gt, resolve_local_flights,
    convert_results_to_metrics_format, measure_predictor_complexity, 
    display_styled_summary, build_introspection_summary
)
 
# Confirm Project Root.
log(LogLevel.INFO, "introspection_module:imports", "Project Root Set: %s", PROJECT_ROOT)

<INFO ><introspection_module:imports> Project Root Set: /home/maxbusato/Projects/Vision-Based-Sense-and-Avoid


In [2]:
# Choose Which Introspection Module to Run
# --------------------------------------------------
# 0 -> None (Control Condition).
# 1 -> Confidence Threshold (Simple Baseline).
# 2 -> Rolling Confidence Trend / EWMA.
# 3 -> Consecutive-Coast-Count Threshold.
SELECTED_INTROSPECTION_MODULE = 1
module_list = ["none", "confidence_threshold", "rolling_confidence_trend", "coast_count_threshold"]

# Evaluation Filtering.
MIN_SCORE = 0.60
DETECTION_THRESHOLD = 0.60
MIN_TRACK_LEN = 8

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/training_subsets/test.txt"
DATASET_PARTS = ["part1", "part2", "part3"]
DATASET_ROOT = os.path.join(PROJECT_ROOT, "data", "corruption_dataset")

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = "outputs/results/introspection_module/" + module_list[SELECTED_INTROSPECTION_MODULE]
run_name = "run3"
#run_name = get_next_run_dir(RESULTS_DIR)

# Evaluation Output Folders.
EVAL_GT_DIR = os.path.join(RESULTS_DIR, run_name, "gt")
EVAL_RESULT_DIR  = os.path.join(RESULTS_DIR, run_name, "result")
EVAL_SUMMARY_DIR = os.path.join(RESULTS_DIR, run_name, "summaries")

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:config", "Configuration Loaded.")
print(f"  Introspection Module : {module_list[SELECTED_INTROSPECTION_MODULE]}")
print(f"  Dataset Root         : {DATASET_ROOT}")
print(f"  Dataset Path         : {DATASET_PATH}")
print(f"  Dataset Parts        : {DATASET_PARTS}")
print(f"  Run Name             : {run_name}")
print(f"  Results Path         : {RESULTS_DIR}")
print(f"  Min Score            : {MIN_SCORE}")
print(f"  Min Track Len        : {MIN_TRACK_LEN}")

<INFO ><introspection_module:config> Configuration Loaded.
  Introspection Module : confidence_threshold
  Dataset Root         : /home/maxbusato/Projects/Vision-Based-Sense-and-Avoid/data/corruption_dataset
  Dataset Path         : configs/datasets/training_subsets/test.txt
  Dataset Parts        : ['part1', 'part2', 'part3']
  Run Name             : run3
  Results Path         : outputs/results/introspection_module/confidence_threshold
  Min Score            : 0.6
  Min Track Len        : 8


In [ ]:
# Download the Required Flight Sequences if Not Already Stored Locally.
downloadDataset(DATASET_PATH)

In [ ]:
# Corrupt the Original Dataset into the Introspection Dataset.
corruptDataset()


---

<br/>

## **5. Data Preparation & Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**5.1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**5.2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [ ]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, DATASET_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "introspection_module:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "introspection_module:preprocessing", " ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, DATASET_ROOT)
log(LogLevel.INFO, "introspection_module:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

# Build Set of Image Filenames That Physically Exist on Disk.
all_images = {name for f in local_flights.values() for name in f["image_names"]}

# Cull GT Entities to Only Those With a Corresponding Image on Disk.
before = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

for flight_hash, flight_data in combined_gt["samples"].items():
    flight_data["entities"] = [
        e for e in flight_data["entities"]
        if e["img_name"] in all_images
    ]

after = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

# Report How Many Images were Removed. 
log(LogLevel.INFO, "introspection_module:preprocessing",
    "GT Culled: %d -> %d entities (%d removed for missing images).", before, after, before - after)

# Save the Filtered Ground Truth to the Evaluation Folder.
os.makedirs(EVAL_GT_DIR, exist_ok=True)
gt_out_path = os.path.join(EVAL_GT_DIR, "groundtruth.json")

# Copy the Ground Truth Values to the .json.
with open(gt_out_path, "w") as f:
    json.dump(combined_gt, f)

# Print Success Message.
log(LogLevel.INFO, "introspection_module:preprocessing", "Ground Truth Saved: %s", gt_out_path)


---

<br/>

## **6. Evaluate Models**

### **6.1. Run Inference**

This section runs inference for the selected backbone configuration. The evaluation is controlled via the `SELECTED_MODEL` parameter, which determines which backbone is instantiated and evaluated within the shared detection pipeline. All models are evaluated under a consistent inference setup to ensure fair comparison across backbone architectures, using the same preprocessing, detection pipeline, and evaluation metrics.

<br/>

#### **6.1.1. Prepare Evaluation Directory & Configure Environment Variables.**

In [ ]:
# Create a Directory where all Evaluation Flight Data will be Stored.
EVAL_DATA_DIR = os.path.join(RESULTS_DIR, run_name, "data")
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

# Link Each Local Flight's Image Folder into the Evaluation Directory.
for folder_name, info in local_flights.items():
    link_path = os.path.join(EVAL_DATA_DIR, folder_name)
    if not os.path.exists(link_path):
        os.symlink(info["images_dir"], link_path)

# Set Environment Variables that AirbornePredictor and SelectedModelPredictor Read.
os.environ["TEST_DATASET_PATH"] = EVAL_DATA_DIR
os.environ["INFERENCE_OUTPUT_PATH"] = RESULTS_DIR
os.environ["DATASET_ENV"] = run_name
os.environ["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
os.environ["MIN_SCORE"] = str(MIN_SCORE)
os.environ["DETECTION_THRESHOLD"] = str(DETECTION_THRESHOLD)

# Disable the Per-Flight Timeout.
os.environ["INFERENCE_PER_FLIGHT_TIMEOUT_SECONDS"] = "0"

# Limit Inference to Only the Specified Flights.
os.environ["PARTIAL_RUN_FLIGHTS"] = ",".join(local_flights.keys())

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Environment Configured.")
print(f"  Test Data Path     : {EVAL_DATA_DIR}")
print(f"  Flights to Process : {len(local_flights)}")
print(f"  MIN_SCORE          : {MIN_SCORE}")
print(f"  MIN_TRACK_LEN      : {MIN_TRACK_LEN}")


<br/>

#### **6.1.2. Run Model Inference Across All Evaluation Flights Using the Selected Configuration.**

> ⚠️ ***This cells requires a CUDA GPU and will take several hours to complete.*** *Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.*

In [ ]:
# Import the Introspective Predictor Class & the Introspection Module Registry.
from src.introspection_module.hrnetw32_introspective.hrnetw32_introspective_test import HRNetW32IntrospectivePredictor
from src.introspection_module.hrnetw32_introspective.introspection_modules import INTROSPECTION_MODULE_MAP

# Check GPU is CUDA Enabled.
if torch.cuda.is_available():
    log(LogLevel.INFO, "introspection_module:device_check", "CUDA-Enabled GPU is Available.")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}.")
    print(f"  CUDA Version: {torch.version.cuda}.")
else:
      log(LogLevel.WARNING, "introspection_module:device_check", "No CUDA-Enabled GPU Found, Using CPU Instead.")

In [ ]:
# Validate Selection.
if SELECTED_INTROSPECTION_MODULE not in INTROSPECTION_MODULE_MAP:
    log(LogLevel.ERROR, "introspection_module:inference", "Invalid SELECTED_INTROSPECTION_MODULE: %s.", SELECTED_INTROSPECTION_MODULE)
    raise ValueError("Please Select a Valid Introspection Module Index.")

# Instantiate the Selected Introspection Module.
introspection_module = INTROSPECTION_MODULE_MAP[SELECTED_INTROSPECTION_MODULE]()

# Initialise the Predictor & Inject the Selected Introspection Module
# BEFORE inference_setup() Runs.
predictor = HRNetW32IntrospectivePredictor()
predictor.introspection_module = introspection_module

# Load Models Before Measuring Complexity.
predictor.inference_setup()

# Measure the Current Model's Complexity.
print()
complexity = measure_predictor_complexity(predictor)

# Run Inference Over all Flights.
print()
predictor.run()

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Inference Complete.")
print(f"  Raw Results Saved to: {os.path.join(RESULTS_DIR, run_name)}\n")

# Set Results Directories. 
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")
METRICS_RESULTS_PATH = os.path.join(EVAL_RESULT_DIR, "result.json")

# Ensure Output Directory Exists.
os.makedirs(EVAL_RESULT_DIR, exist_ok=True)

# Copy the Results File.
convert_results_to_metrics_format(RAW_RESULTS_PATH, METRICS_RESULTS_PATH)

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Results Copied to Metrics Directory.")
print(f"  Metrics-Ready Results Stored at : {METRICS_RESULTS_PATH}")


---

<br/>

## **7. Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Locate the Competition Metrics Script.
metrics_script = os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py")

# Call the Competition Metrics Script.
!python3 {metrics_script} \
    --dataset-folder   {EVAL_GT_DIR} \
    --results-folder   {EVAL_RESULT_DIR} \
    --summaries-folder {EVAL_SUMMARY_DIR} \
    --min-score        {MIN_SCORE} \
    --min-track-len    {MIN_TRACK_LEN}

# Print Message on Completion. 
log(LogLevel.INFO, "introspection_module:evaluation", "Metrics Complete.")


---

<br/>

## **8. Benchmarking Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide a measure of performance on the full hardened dataset. Additionally, a random sample of results is generated using both the ground truth bounding boxes and the predicted ones, enabling a visual analysis of the model’s performance.

In [ ]:
# Find the Summary JSON in the Summaries Directory.
summary_files = glob(os.path.join(EVAL_SUMMARY_DIR, "*.json"))

# Check that the Summary Files Exist.
if not summary_files:
    log(LogLevel.ERROR, "introspection_module:summary", "No Summary Files Found! Run Inference First.")
else:
    # Build Summary DataFrame.
    summary_df = build_introspection_summary(
        summary_json_path = summary_files[0],
        run_dir           = os.path.join(RESULTS_DIR, run_name),
        output_csv_path   = os.path.join(EVAL_SUMMARY_DIR, "introspection_module_benchmark_summary.csv"),
    )

    # Print Results.
    display_styled_summary(summary_df)


---

<br/>

## **9. Introspection Module Evaluation Results**

The following results compare the previously described introspection strategies when integrated with the HRNet-W32 Tracking-to-Detect (T2D) backbone. Performance is reported on the held-out test set (243 flights), excluding the hardened training subset used during model development. This test set comprises all remaining unseen evaluation data.

<div style="background-color:#fff8dc; border-left:5px solid #f0c419; padding:12px 16px; margin:10px 0;">
<strong>⚠️ PLACEHOLDER - Results Pending Final Inference Runs.</strong><br/>
Populate this table after all introspection modules have been evaluated across the all datasets and the corresponding metrics have been collected. This represents the final stage of the introspection evaluation.
</div>


---

<br/>

## **10. Introspection Module Visualisations**

To complement the aggregate metrics in Sections 8 and 9, this section renders per-flight annotated videos showing exactly how each introspection module tracked a sequence over time, and where it chose to raise a warning. Rather than judging a module purely on summary statistics, this gives a frame-by-frame view of tracking coverage and warning behaviour across a full flight. This is also useful for sanity-checking that a module which looks strong on paper is also behaving sensibly in practice.

In [3]:
# Set Visualisations Folder Path. 
VISUALISER_DIR = os.path.join(RESULTS_DIR, run_name, "visualisations")

# Ensure Visualisations Directory Exists.
os.makedirs(VISUALISER_DIR, exist_ok=True)

In [4]:
# Load the AOT Dataset Object & Corruption Manifest -- Needed for the
# Ground-Truth Overlay and per-Frame Severity Shown in Generated Videos.
from src.data.original_dataset.browse_data import dataLoader

dataset = dataLoader(download=False)

# Build a flight_id -> Manifest Entry Lookup. Returns None for Clean
# (Uncorrupted) Flights -- generateIntrospectionVideo() Handles that
# Correctly by Simply Omitting the Severity Display.
CORRUPTION_MANIFEST_PATH = "configs/datasets/corruption_subsets/corruption_manifest.json"

with open(CORRUPTION_MANIFEST_PATH, "r") as f:
    corruption_manifest = json.load(f)

manifest_lookup = {entry["flight_id"]: entry for entry in corruption_manifest["flights"]}

log(LogLevel.INFO, "introspection_module:visualisation_setup",
    "Loaded Corruption Manifest: %d Corrupted Flight(s) Available for Severity Overlay.",
    len(manifest_lookup))


<INFO ><browse_data:dataLoader> Loading Dataset Metadata.



2026-09-06 11:13:02.563 | INFO     | core.dataset:load_ve:36 - Loading valid encounters...
2026-09-06 11:13:02.613 | INFO     | core.dataset:load_gt:20 - Loading ground truth...
2026-09-06 11:13:09.682 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present in valid encounters: 673f29c3e4b4428fa26bc55d812d45d9
2026-09-06 11:13:09.683 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present in valid encounters: 63c5597a57b04b448723f1f1844a2b78
2026-09-06 11:13:09.685 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present in valid encounters: 4458bb328345402c8f363b7480060393
2026-09-06 11:13:09.695 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present in valid encounters: 7f5dd587c1b046efbf9e3221fe633e4e
2026-09-06 11:13:09.697 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present in valid encounters: 3c349bdf14c1464bb2b30094e7297d84
2026-09-06 11:13:09.698 | INFO     | core.dataset:load_gt:30 - Skipping flight, not present 


<INFO ><browse_data:dataLoader> Dataset Metadata Loaded Successfully.
<INFO ><introspection_module:visualisation_setup> Loaded Corruption Manifest: 225 Corrupted Flight(s) Available for Severity Overlay.


<br/>

### **10.1. Generate Per-Flight Prediction Videos**

For a selected flight, `generateIntrospectionVideo()` walks every frame of the flight in order *(not just the frames a track happened to be present for)* and overlays that module's predicted detections as bounding boxes, colour-coded by track state *(active detection, coasted, or introspection warning)*. A status banner along the bottom tracks frame progress and escalates colour whenever the module raises a warning, so degraded sequences are immediately visible when scrubbing through the video.

#### **Static Confidence Trend**

In [5]:
# Static Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part2acfeb72c54cc4733a31c32fd1c14df7f", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part2acfeb72c54cc4733a31c32fd1c14df7f", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part2acfeb72c54cc4733a31c32fd1c14df7f.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part2acfeb72c54cc4733a31c32fd1c14df7f")
)

# Static Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part154a62b2bceba487dbaee2c5d959ea25b", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part154a62b2bceba487dbaee2c5d959ea25b", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part154a62b2bceba487dbaee2c5d959ea25b.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part154a62b2bceba487dbaee2c5d959ea25b")
)

# Static Confidence Trend (Failing FNR Example).
generateIntrospectionVideo(
    flight_id        = "part38640ad5643e94cd8a921cf263aade59a", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part38640ad5643e94cd8a921cf263aade59a", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part38640ad5643e94cd8a921cf263aade59a.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part38640ad5643e94cd8a921cf263aade59a")
)

<INFO ><introspection_visualiser:resolveFlightImagePaths> Found 183 Frames for Flight part2acfeb72c54cc4733a31c32fd1c14df7f in 'data/original_dataset/part2/Images/part2acfeb72c54cc4733a31c32fd1c14df7f'.
<INFO ><introspection_visualiser:loadIntrospectionResults> Loaded Results for 73 Frames from 'outputs/results/introspection_module/confidence_threshold/run3/part2acfeb72c54cc4733a31c32fd1c14df7f/result.json'.
<INFO ><introspection_visualiser:loadGroundTruthForFlight> Loaded Ground Truth for Flight part2acfeb72c54cc4733a31c32fd1c14df7f: 183 / 183 Frames have a Target.
<INFO ><introspection_visualiser:generateIntrospectionVideo> Generating 'Static Confidence Trend' Introspection Video for Flight part2acfeb72c54cc4733a31c32fd1c14df7f  (183 Frames).
<INFO ><introspection_visualiser:generateIntrospectionVideo> Writing 183 Frames at 2448x2048  10 fps.
<INFO ><introspection_visualiser:generateIntrospectionVideo> Raw Video Written (183 / 183 Frames). Re-encoding to H.264.
<INFO ><introspection_

#### **Rolling Confidence Trend**

In [ ]:
# Rolling Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part2acfeb72c54cc4733a31c32fd1c14df7f", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part2acfeb72c54cc4733a31c32fd1c14df7f", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part2acfeb72c54cc4733a31c32fd1c14df7f.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part2acfeb72c54cc4733a31c32fd1c14df7f")
)

# Rolling Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part154a62b2bceba487dbaee2c5d959ea25b", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part154a62b2bceba487dbaee2c5d959ea25b", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part154a62b2bceba487dbaee2c5d959ea25b.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part154a62b2bceba487dbaee2c5d959ea25b")
)

# Rolling Confidence Trend (Failing FNR Example).
generateIntrospectionVideo(
    flight_id        = "part38640ad5643e94cd8a921cf263aade59a", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part38640ad5643e94cd8a921cf263aade59a", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part38640ad5643e94cd8a921cf263aade59a.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part38640ad5643e94cd8a921cf263aade59a")
)

<br/>

### **10.2. Side-by-Side Module Comparison**

`spliceVideosSideBySide()` takes the per-flight videos produced above and composites them into a single comparison video, played back in sync. This makes it straightforward to directly compare how two introspection strategies handle the *same* flight. For example, whether a more conservative module holds the track for longer, or raises a warning earlier than a simpler threshold-based one.

In [ ]:
# Combine Static & Rolling Confidence Flights (Successful Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/successful_warning_example_1.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/successful_warning_example_1.mp4",
    ],
    output_path = "docs/Videos/successful_warning_static_vs_rolling_1.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

# Combine Static & Rolling Confidence Flights (Successful Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/successful_warning_example_2.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/successful_warning_example_2.mp4",
    ],
    output_path = "docs/Videos/successful_warning_static_vs_rolling_2.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

# Combine Static & Rolling Confidence Flights (Failing FNR Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/failing_warning_example.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/failing_warning_example.mp4",
    ],
    output_path = "docs/Videos/failing_warning_static_vs_rolling.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

---